# 04 — Rendering owlready2 ontologies back to Manchester syntax

`pymos` is symmetric: the same package that **parses** a Manchester `.omn`
document into owlready2 will also **render** an owlready2 ontology back to a
Manchester document — full round-trip, pure Python, no Java.

This notebook covers the three public rendering entry points:

1. `pymos.render_expression(ce, prefixes)` — one class expression to a string,
   precedence-aware (parentheses are inserted around lower-precedence operands).
2. `pymos.render_frame(entity, prefixes)` — one Class / ObjectProperty /
   DataProperty / Individual / AnnotationProperty frame.
3. `pymos.render(onto, prefixes)` — a full document with Prefix declarations,
   Ontology header, and frames in stable order.

In [1]:
from pathlib import Path

import pymos

print("pymos version:", pymos.__version__)

pymos version: 0.1.0


## A. Render a single class expression

`render_expression` is the inverse of `parse_expression`. It takes any
owlready2 class expression (a `Restriction`, `And`, `Or`, `Not`, `OneOf`,
`ConstrainedDatatype`, or a named class) and produces a Manchester string.

Precedence is handled automatically: `or` binds looser than `and`, which binds
looser than `not`, so an `or` operand inside an `and` gets parenthesised.

In [2]:
import owlready2

w = owlready2.World()
onto = w.get_ontology("http://example.org/biomed#")
with onto:
    class treats(owlready2.ObjectProperty): pass
    class Disease(owlready2.Thing): pass
    class BacterialInfection(Disease): pass
    class ViralInfection(Disease): pass

PFX = {"": "http://example.org/biomed#"}

# A simple existential restriction
ce1 = pymos.parse_expression("treats some BacterialInfection", onto, prefixes=PFX)
print(pymos.render_expression(ce1, prefixes=PFX))

# Boolean nesting — note the parentheses around the `or`
ce2 = pymos.parse_expression(
    "treats some (BacterialInfection or ViralInfection)", onto, prefixes=PFX
)
print(pymos.render_expression(ce2, prefixes=PFX))

# Cardinality
ce3 = pymos.parse_expression("treats min 2 Disease", onto, prefixes=PFX)
print(pymos.render_expression(ce3, prefixes=PFX))

:treats some :BacterialInfection
:treats some (:BacterialInfection or :ViralInfection)
:treats min 2 :Disease


## B. Render a single frame

`render_frame` emits the header line and every populated axiom keyword for
one entity. Empty axiom groups are omitted, so a class with no asserted
axioms produces just `Class: Foo`.

In [3]:
doc = """
Prefix: : <http://example.org/biomed#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: Antibiotic
    Annotations: rdfs:label "Antibiotic"
    SubClassOf: Drug
    EquivalentTo: Drug and (treats some BacterialInfection)

ObjectProperty: treats
    Domain: Drug
    Range: Disease
    Characteristics: Transitive

Individual: penicillin
    Types: Antibiotic
    Facts: treats infection1
"""

PFX2 = {
    "": "http://example.org/biomed#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
}

onto2 = pymos.parse(doc)

antibiotic = onto2.world["http://example.org/biomed#Antibiotic"]
print(pymos.render_frame(antibiotic, prefixes=PFX2))

Class: :Antibiotic
    Annotations: rdfs:label "Antibiotic"
    SubClassOf: :Drug
    EquivalentTo: :Drug and :treats some :BacterialInfection



In [4]:
treats = onto2.world["http://example.org/biomed#treats"]
print(pymos.render_frame(treats, prefixes=PFX2))

ObjectProperty: :treats
    Domain: :Drug
    Range: :Disease
    Characteristics: Transitive



In [5]:
penicillin = onto2.world["http://example.org/biomed#penicillin"]
print(pymos.render_frame(penicillin, prefixes=PFX2))

Individual: :penicillin
    Types: :Antibiotic
    Facts: :treats :infection1



## C. Render a full document

`pymos.render(onto, prefixes)` assembles the full Manchester document:
Prefix declarations, Ontology header, then frames in stable order
(Datatype → AnnotationProperty → ObjectProperty → DataProperty → Class →
Individual, each sorted by IRI). The output is deterministic — running
`render` on the same ontology twice yields byte-identical text.

Below we load the `biomed.omn` fixture used by notebooks 02 and 03, and
render the resulting owlready2 ontology back to Manchester.

In [6]:
OMN = Path("..") / "data" / "biomed.omn"
biomed_doc = OMN.read_text()
biomed_onto = pymos.parse(biomed_doc)

rendered = pymos.render(biomed_onto, prefixes=PFX2)
print(rendered)

Prefix: : <http://example.org/biomed#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>
Ontology: <http://example.org/biomed>

ObjectProperty: :associatedWith

ObjectProperty: :causes

ObjectProperty: :hasTarget

ObjectProperty: :treats
    Domain: :Drug
    Range: :Disease

Class: :Antibiotic
    Annotations: rdfs:label "Antibiotic"
    SubClassOf: :Drug
    EquivalentTo: :Drug and :treats some :BacterialInfection

Class: :Antiviral
    Annotations: rdfs:label "Antiviral"
    SubClassOf: :Drug
    EquivalentTo: :Drug and :treats some :ViralInfection

Class: :BacterialInfection
    Annotations: rdfs:label "Bacterial Infection"
    SubClassOf: :InfectiousDisease

Class: :BiologicalEntity
    Annotations: rdfs:label "Biological Entity"

Class: :ChemicalEntity
    Annotations: rdfs:label "Chemical Entity"

Class: :Disease
    Annotations: rdfs:label "Disease"
    SubClassOf: :BiologicalEntity

Class: :Drug
    Annotations: rdfs:label "Drug"
    SubClassOf: :ChemicalEntity

Class: :Ge

## D. The round-trip contract

Two guarantees:

- **Structural equality**: `parse → render → parse` preserves the set of
  class / property / individual IRIs and the count of axioms per entity.
- **Idempotency**: the second pass of `render` produces byte-identical text.

(The first pass is *not* guaranteed byte-identical to the source — the
source may have differing whitespace, frame ordering, or use prefixes that
pymos doesn't have in its map. The second pass stabilises.)

In [7]:
text1 = pymos.render(pymos.parse(biomed_doc), prefixes=PFX2)
text2 = pymos.render(pymos.parse(text1), prefixes=PFX2)

print("byte-identical second pass?", text1 == text2)

byte-identical second pass? True


In [8]:
onto_a = pymos.parse(biomed_doc)
onto_b = pymos.parse(text1)

iris_a = {c.iri for c in onto_a.classes()}
iris_b = {c.iri for c in onto_b.classes()}

print("class IRIs preserved?", iris_a == iris_b)
print("number of classes:", len(iris_a))

class IRIs preserved? True
number of classes: 11


## E. What `render` covers

As of this notebook, `pymos.render` emits:

- **Header**: `Prefix:` declarations (from the prefix map you pass) and
  `Ontology:` IRI; `Import:` directives if any are present.
- **Datatype** frames for any IRI declared as `rdfs:Datatype`.
- **AnnotationProperty** frames.
- **ObjectProperty / DataProperty** frames with `Domain:`, `Range:`,
  `Characteristics:` (`Functional`, `InverseFunctional`, `Transitive`,
  `Symmetric`, `Asymmetric`, `Reflexive`, `Irreflexive`), `SubPropertyOf:`,
  and `InverseOf:`.
- **Class** frames with `Annotations:` (built-in `rdfs:label`/`rdfs:comment`/
  `rdfs:seeAlso` and user-declared annotation properties), `SubClassOf:`,
  `EquivalentTo:`, and `DisjointWith:`.
- **Individual** frames with `Annotations:`, `Types:`, `Facts:` (object and
  data property assertions), `SameAs:`, and `DifferentFrom:`.

Class expressions inside any of those keywords are rendered with operator
precedence preserved through parentheses.

## Takeaway

`pymos.render` closes the loop: parse a `.omn` file, manipulate the
ontology programmatically through owlready2, then re-emit a clean,
deterministic Manchester document — all without leaving Python.